# Day 8: Hash-Based Sharding
**Situation:** We try to distribute evenly using a hashing function.
**Problem:** What should we hash? user_id? channel_id? message_id? We have to make a choice.

In [ ]:
class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content

class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def stats(self):
        for shard in self.shards:
            print(f"Shard {shard.id}: {len(shard.messages)} messages")

In [ ]:
import hashlib

class HashShardManager(ShardManager):

    def get_shard(self, key):
        # MD5 hashing the key and mapping to a shard
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return self.shards[h % len(self.shards)]

    def send_message(self, message):
        # DECISION: What should be hashed?
        # Option A: message.user_id -> Hotspot if one user spams.
        # Option B: message.channel_id -> Hotspot if channel goes viral.
        # Option C: message content/id -> Perfect distribution, but reading a channel requires cross-shard query.
        
        # We will use a unique mix of the message's content, user, and channel as a proxy for an internal message_id. 
        # This acts effectively like a random hashing key per message!
        message_key = f"{message.user_id}-{message.channel_id}-{message.content}-{random.random()}"
        shard = self.get_shard(message_key)
        shard.store(message)

### Simulation: Testing Hash-Based Sharding

In [ ]:
import random
manager = HashShardManager(3)

# Viral Event on Channel 99 with heavily active users
for _ in range(6000):
        msg = Message(user_id=random.randint(1, 5), channel_id=99, content="Spamming the chat!")
        manager.send_message(msg)

manager.stats()

### Rationale for Hash Key Choice
I chose to generate a unique key effectively simulating a random `message_id`. 

**Justification:** The primary issue with User and Channel sharding was hotspotting. By hashing a unique message identifier, we guarantee that messages are spread perfectly across all available servers regardless of whether a single user or channel dominates the traffic. 

**Tradeoff:** The immense cost of this perfectly even distribution is that messages for the same channel are now fragmented across all 3 shards. When we need to fetch a channel's history, we can no longer query just one server. This introduces 'Scatter-Gather' querying.